# Exploración: perfiles públicos de Instagram por username

Playwright **sin login y sin LLM**. Cada perfil se carga **una sola vez** y se lee de tres formas: metadatos, contenido visible y JSON incrustado.

- Chrome normal y limpio: no disfraza el navegador, no llama endpoints internos ni resuelve captchas.
- Ante un bloqueo **se detiene**, sin reintentar.
- Las fotos van a `files/instagram/` (ignorado por git).
- Las salidas contienen datos personales: límpialas antes de hacer commit.

In [1]:
import os, sys, re, io, json, random, asyncio
from pathlib import Path
from urllib.parse import urlsplit

if not os.path.isdir("app"):
    os.chdir(os.path.abspath(".."))  # raíz del repo
sys.path.insert(0, os.getcwd())

from PIL import Image
from playwright.async_api import async_playwright
from app.infrastructure.outbound.extractors.shared.browser import run_isolated  # solo el bucle asyncio

CASOS = {"nasa": "OK", "cuenta_publica": "OK", "cuenta_privada": "PRIVADO", "cuenta_inexistente": "NO_DATA"}
DESTINO = Path("files/instagram")
USERNAME_RE = re.compile(r"^(?!.*\.\.)(?!\.)[a-z0-9._]{1,30}(?<!\.)$")

def normalizar(valor):
    v = valor.strip().lower()
    m = re.match(r"^(?:https?://)?(?:www\.)?instagram\.com/([^/?#]+)/?(?:[?#].*)?$", v)
    v = (m.group(1) if m else v).lstrip("@")
    if not USERNAME_RE.match(v):
        raise ValueError(f"Username inválido: {valor!r}")
    return v

for v in ("@nasa", "https://www.instagram.com/nasa/?hl=es"):
    assert normalizar(v) == "nasa"

In [ ]:
JS_PAGINA = """() => {
  const metas = {};
  for (const e of document.querySelectorAll('meta[property], meta[name]'))
    metas[e.getAttribute('property') || e.getAttribute('name')] = e.content;
  const main = document.querySelector('main');
  return {
    url: location.href, title: document.title, metas,
    body: document.body.innerText,
    main: main ? main.innerText : '',
    password: !!document.querySelector('input[type=password]'),
    imgs: [...document.querySelectorAll('img')].map(i => ({alt: i.alt, src: i.currentSrc || i.src, w: i.naturalWidth})),
    json: [...document.querySelectorAll('script[type="application/json"]')].map(s => s.textContent),
  };
}"""

NO_DISPONIBLE = re.compile(r"(?:página|perfil|profile) no está disponible|page isn.t available|profile isn.t available", re.I)
PRIVADA = re.compile(r"(?:esta )?cuenta (?:es )?privada|account is private|this profile is private|perfil (?:es )?privado", re.I)
DETENIDO = re.compile(r"espera unos minutos|wait a few minutes|no eres un robot|not a robot|captcha|actividad sospechosa", re.I)

def clasificar(p, username):
    ruta = urlsplit(p["url"]).path
    if p["status"] in (401, 403, 429) or ruta.startswith(("/accounts/", "/challenge")) or (DETENIDO.search(p["body"]) and not p["main"]):
        return "BLOQUEADO"
    if NO_DISPONIBLE.search(p["title"]):  # el título: una biografía podría contener la frase
        return "NO_DATA"
    if ruta.strip("/").lower() != username:
        return "ERROR"
    if not p["main"]:
        return "BLOQUEADO" if p["password"] else "ERROR"
    return "PERFIL"

def leer_metas(p):
    titulo = p["metas"].get("og:title", "")
    m = re.match(r"^(?P<nombre>.*?)\s*\(@(?P<user>[\w.]+)\)", titulo)
    desc = p["metas"].get("description", "")
    b = re.search(r'(?:en|on) Instagram: "(.*)"\s*$', desc, re.S)
    return {"username": m["user"].lower() if m else None, "full_name": (m["nombre"] or None) if m else None,
            "biography": b.group(1) if b else None, "photo_url": p["metas"].get("og:image")}

def leer_visible(p, username):
    lineas = [l.strip() for l in p["main"].splitlines() if l.strip()]
    foto = next((i for i in p["imgs"] if re.search(r"perfil|profile", i["alt"], re.I) and username in i["alt"].lower()), None)
    return {"username": username if username in [l.lower() for l in lineas] else None,
            "photo_url": foto["src"] if foto else None,
            "is_private": bool(PRIVADA.search(p["main"])), "lineas": lineas[:25]}

def leer_json(p, username):
    mejor = {}
    def andar(o):
        nonlocal mejor
        if isinstance(o, dict):
            if str(o.get("username", "")).lower() == username and "biography" in o:  # sin biography es enrutamiento
                mejor = o
            for x in o.values():
                andar(x)
        elif isinstance(o, list):
            for x in o:
                andar(x)
    for t in p["json"]:
        if username in t.lower():
            try:
                andar(json.loads(t))
            except ValueError:
                pass
    if not mejor:
        return None
    foto = (mejor.get("hd_profile_pic_url_info") or {}).get("url") or mejor.get("profile_pic_url")
    return {"username": username, "full_name": mejor.get("full_name"), "biography": mejor.get("biography"),
            "is_private": mejor.get("is_private"), "photo_url": foto}

def consolidar(p, username):
    """JSON principal; solo es OK si username, nombre y biografía cuadran con los metadatos."""
    estado = clasificar(p, username)
    if estado != "PERFIL":
        return estado, None
    j, m, v = leer_json(p, username), leer_metas(p), leer_visible(p, username)
    limpio = lambda t: " ".join((t or "").split())
    if not j or not j["photo_url"] or j["is_private"] is None or m["username"] != username:
        return "ERROR", None
    if limpio(j["full_name"]) != limpio(m["full_name"]) or j["is_private"] != v["is_private"]:
        return "ERROR", None
    if m["biography"] is not None and limpio(j["biography"]) != limpio(m["biography"]):
        return "ERROR", None
    return ("PRIVADO" if j["is_private"] else "OK"), j

In [ ]:
async def guardar_foto(context, url, username):
    host = urlsplit(url or "").hostname or ""
    if not host.endswith((".cdninstagram.com", ".fbcdn.net")):
        return {"error": f"host no permitido: {host}"}
    r = await context.request.get(url)
    if not r.ok:
        return {"error": f"HTTP {r.status}"}
    datos = await r.body()
    try:
        with Image.open(io.BytesIO(datos)) as im:
            im.load()
            formato, tam = im.format, im.size
    except Exception as e:
        return {"error": f"no es imagen: {e}"}
    DESTINO.mkdir(parents=True, exist_ok=True)
    ruta = DESTINO / f"{username}.{formato.lower()}"
    ruta.write_bytes(datos)
    return {"ruta": ruta.as_posix(), "formato": formato, "tam": tam}

async def visitar(usernames, headless):
    res = {}
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(channel="chrome", headless=headless)
        try:
            for i, u in enumerate(usernames):
                if i:
                    await asyncio.sleep(random.uniform(10, 15))
                context = await browser.new_context(locale="es-PE", viewport={"width": 1366, "height": 768})
                try:
                    page = await context.new_page()
                    r = await page.goto(f"https://www.instagram.com/{u}/", wait_until="domcontentloaded", timeout=45000)
                    await page.wait_for_timeout(6000)
                    p = await page.evaluate(JS_PAGINA)
                    p["status"] = r.status if r else None
                    d = {"pagina": p, "estado": clasificar(p, u)}
                    res[u] = d
                    if d["estado"] == "PERFIL":
                        d |= {"metas": leer_metas(p), "visible": leer_visible(p, u), "json": leer_json(p, u)}
                        url = next((x["photo_url"] for x in (d["json"], d["visible"], d["metas"]) if x and x["photo_url"]), None)
                        d["foto"] = await guardar_foto(context, url, u)
                    print(u, "| HTTP", p["status"], "|", d["estado"])
                    if d["estado"] in ("BLOQUEADO", "ERROR"):
                        print("Se detiene sin reintentar.")
                        break
                except Exception as e:
                    res.setdefault(u, {}).update(estado="ERROR", detalle=f"{type(e).__name__}: {e}")
                    print(u, "| ERROR |", e)
                    break
                finally:
                    await context.close()
        finally:
            await browser.close()
    return res

def mostrar(res):
    for u, d in res.items():
        print("=" * 20, u, "| esperado:", CASOS[u], "| obtenido:",
              consolidar(d["pagina"], u)[0] if "pagina" in d else d["estado"], d.get("detalle", ""))
        for k in ("metas", "visible", "json", "foto"):
            if k in d:
                print(k, "->", json.dumps(d[k], ensure_ascii=False, indent=1))

In [ ]:
# 1) Ventana visible: 4 visitas con pausa
resultados = await run_isolated(lambda: visitar([normalizar(u) for u in CASOS], headless=False))
mostrar(resultados)

In [ ]:
# 2) Modo oculto: una sola visita, y solo si todo lo anterior cargó
if len(resultados) == len(CASOS) and all(d["estado"] in ("PERFIL", "NO_DATA") for d in resultados.values()):
    oculto = await run_isolated(lambda: visitar(["nasa"], headless=True))
    mostrar(oculto)
else:
    print("Omitido: la prueba visible no terminó bien.")